In [10]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Cell 2: Load IsoDecipher count matrix
counts = pd.read_csv('../results/counts/exp93_isoform_count.csv', index_col=0)
counts.columns = counts.columns.str.replace('_nan$', '', regex=True)
print(f"Matrix: {counts.shape[0]} cells × {counts.shape[1]} isoform features")

# Cell 3: Core metric function
def compute_metric(counts, gene, log_scale=False, min_pct=0.01):
    """
    Auto-select PUI or Entropy based on number of expressed isoform groups.
    
    Parameters:
    -----------
    gene      : str   — gene name e.g. 'IGHM', 'CD59', 'CD44'
    log_scale : bool  — log1p normalization for PUI (recommended for immunoglobulins)
    min_pct   : float — minimum fraction of cells expressing a group to count as expressed
    
    Returns:
    --------
    values : np.array — per-cell PUI or entropy
    method : str      — 'PUI' or 'Entropy'
    """
    cols = [c for c in counts.columns if c.startswith(f'{gene}_G')]
    
    if len(cols) < 2:
        print(f"{gene}: only 1 group found, skipping")
        return None, None
    
    X     = counts[cols].values.astype(float)
    total = X.sum(axis=1)
    
    # Count meaningfully expressed groups
    pct_expressing  = (X > 0).mean(axis=0)
    expressed_groups = (pct_expressing > min_pct).sum()
    
    if expressed_groups <= 2:
        # PUI — use two most expressed groups
        dominant = np.argsort(pct_expressing)[-2:]
        g0 = X[:, dominant[0]]
        g1 = X[:, dominant[1]]
        if log_scale:
            g0, g1 = np.log1p(g0), np.log1p(g1)
        denom = g0 + g1
        values = np.where(denom > 0, g0 / (denom + 1e-10), np.nan)
        return values, 'PUI'
    else:
        # Entropy — no log needed, probability is scale-free
        p      = X / (total[:, np.newaxis] + 1e-10)
        values = -(p * np.log(p + 1e-10)).sum(axis=1)
        values = np.where(total > 0, values, np.nan)
        return values, 'Entropy'

# Cell 4: Example usage
pui_ighm,  m1 = compute_metric(counts, 'IGHM', log_scale=True)   # IG
pui_cd59,  m2 = compute_metric(counts, 'CD59')                    # 2-group
ent_cd44,  m3 = compute_metric(counts, 'CD44')                    # multi-group

print(f"IGHM → {m1}: mean={np.nanmean(pui_ighm):.3f}")
print(f"CD59 → {m2}: mean={np.nanmean(pui_cd59):.3f}")
print(f"CD44 → {m3}: mean={np.nanmean(ent_cd44):.3f}")



Matrix: 4630 cells × 1369 isoform features
IGHM → PUI: mean=0.295
CD59 → Entropy: mean=0.129
CD44 → Entropy: mean=0.143


In [ ]:
# Cell 5: Load into AnnData
adata_iso = sc.read_csv('../results/counts/exp93_isoform_count.csv')
adata_iso.var_names = adata_iso.var_names.str.replace('_nan$', '', regex=True)

print(f"adata_iso: {adata_iso.shape}")  
print(f"obs (cells): {adata_iso.obs_names[:3].tolist()}")
print(f"var (features): {adata_iso.var_names[:3].tolist()}")

# Then add metrics
adata_iso.obs['IGHM_PUI']     = pui_ighm
adata_iso.obs['CD59_PUI']     = pui_cd59
adata_iso.obs['CD44_entropy'] = ent_cd44
print(adata_iso)

adata_iso: (4630, 1369)
obs (cells): ['AAACCCACAAATAGCA-1', 'AAACCCAGTATCGGTT-1', 'AAACCCAGTGACACGA-1']
var (features): ['ABL1_G0', 'ACTB_G0', 'ACTB_G1']
AnnData object with n_obs × n_vars = 4630 × 1369
    obs: 'IGHM_PUI', 'CD59_PUI', 'CD44_entropy'


In [ ]:
# Cell 6: Integrate with GEX AnnData
import anndata as ad

# ── Option A: Add PUI/Entropy metrics to existing GEX AnnData obs ─────
# Lightweight — recommended for pseudotime and visualization workflows

# Find shared cells
shared = adata_gex.obs_names.intersection(adata_iso.obs_names)
print(f"Shared cells: {len(shared)}")

# Add metrics directly to GEX AnnData
adata_gex.obs.loc[shared, 'IGHM_PUI']     = pui_ighm
adata_gex.obs.loc[shared, 'CD59_PUI']     = pui_cd59
adata_gex.obs.loc[shared, 'CD44_entropy'] = ent_cd44

# Now usable in any scanpy plot
sc.pl.umap(adata_gex, color=['IGHM_PUI', 'CD59_PUI', 'CD44_entropy'])

# ── Option B: Full concat — isoform features as additional vars ───────
# Use when you want isoform counts in the feature matrix (e.g. joint PCA)

iso_sub = adata_iso[shared].copy()
gex_sub = adata_gex[shared].copy()

adata_joint = ad.concat(
    [gex_sub, iso_sub],
    axis=1,
    join='outer',
    merge='same'
)
print(f"Joint AnnData: {adata_joint.shape}")
# → (n_cells × (n_gex_genes + n_isoform_features))